Adding Anchor(pos_emb at recursion)

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import GPT2Tokenizer
import math
import copy

# ==========================================
# 1. Diffusion Utilities (Sqrt Schedule)
# ==========================================
def compute_sqrt_alpha_bar(t, T_max=2000, s=1e-4):
    """
    Sqrt noise schedule from Diffusion-LM (Appendix A).
    alpha_bar_t = 1 - sqrt(t/T + s)
    """
    # t is expected to be a tensor of shape (batch_size, 1, 1)
    t_norm = t.float() / T_max
    alpha_bar = 1.0 - torch.sqrt(t_norm + s)
    # Clamp to prevent negative variances or exact zeros
    return torch.clamp(alpha_bar, min=1e-5, max=0.9999)

class EMA:
    """Exponential Moving Average for model weights (Crucial for TRM stability)"""
    def __init__(self, model, decay=0.999):
        self.model = model
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    def update(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                new_average = (1.0 - self.decay) * param.data + self.decay * self.shadow[name]
                self.shadow[name] = new_average.clone()

    def apply_shadow(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data
                param.data = self.shadow[name]

    def restore(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                param.data = self.backup[name]

In [2]:
# ==========================================
# 2. Model Architecture (TRM + Diffusion)
# ==========================================
class BidirectionalTransformerLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.norm1 = nn.LayerNorm(d_model)
        
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        batch_size, seq_len, _ = x.size()
        q = self.q_proj(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)

        attn_out = F.scaled_dot_product_attention(q, k, v, is_causal=False)
        attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)
        
        x = self.norm1(x + self.out_proj(attn_out))
        x = self.norm2(x + self.mlp(x))
        return x

class TRM_Diffusion(nn.Module):
    def __init__(self, vocab_size, d_model=512, n_heads=8, d_ff=2048, num_layers=4):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        
        # End-to-End Embeddings
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.token_emb.weight.requires_grad = True
        self.pos_emb = nn.Embedding(1024, d_model) 
        
        # Timestep embedding
        self.time_mlp = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model)
        )
        
        # TRM Backbone
        self.layers = nn.ModuleList([
            BidirectionalTransformerLayer(d_model, n_heads, d_ff) 
            for _ in range(num_layers)
        ])
        
        self.output_head = nn.Linear(d_model, d_model)
        self.z_init = nn.Parameter(torch.randn(1, 1, d_model))
        self.lm_head = nn.Linear(d_model, vocab_size)

    def get_sinusoidal_embeddings(self, t):
        half_dim = self.d_model // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=t.device) * -emb)
        emb = t.float().unsqueeze(1) * emb.unsqueeze(0)
        emb = torch.cat((emb.sin(), emb.cos()), dim=-1)
        return emb

    def latent_recursion(self, x_t_emb, z, t_emb, n, pos_embeddings):
        for _ in range(n):
            combined_state = x_t_emb + z + t_emb.unsqueeze(1) + pos_embeddings
            for layer in self.layers:
                combined_state = layer(combined_state)
            z = combined_state
            
        pred_x0_emb = self.output_head(z)
        return pred_x0_emb, z

    def deep_recursion(self, x_t_emb, z, t_emb, n, T, pos_embeddings):
        for _ in range(T):
            pred_x0_emb, z = self.latent_recursion(x_t_emb, z, t_emb, n, pos_embeddings)
        return pred_x0_emb, z

    def forward(self, x_t_emb, t, positions, n=6, T=3): 
        # FIX 1: Convert integer timesteps to continuous sinusoidal
        sinusoidal_t = self.get_sinusoidal_embeddings(t)
        t_emb = self.time_mlp(sinusoidal_t)
        
        pos_embeddings = self.pos_emb(positions)
        
        # FIX 2: Dynamically expand z_init using x_t_emb's shape
        bs, seq_len, _ = x_t_emb.shape
        z = self.z_init.expand(bs, seq_len, -1)
        
        pred_x0_emb, z_final = self.deep_recursion(x_t_emb, z, t_emb, n, T, pos_embeddings)
        
        # FIX 3: Return ONLY pred_x0_emb so we don't crash the tuple unpacker
        return pred_x0_emb

In [3]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import PreTrainedTokenizerFast

class TinyStoriesLocalDataset(Dataset):
    def __init__(self, file_path, tokenizer, seq_len=64, limit=1000000):
        self.tokenizer = tokenizer
        self.seq_len = seq_len
        self.stories = []
        
        print(f"Loading up to {limit} stories from {file_path}...")
        with open(file_path, "r", encoding="utf-8") as f:
            for i, line in enumerate(f):
                if i >= limit:
                    break
                
                line = line.strip()
                if line: # Skip empty lines
                    self.stories.append(line)
                    
        print(f"Successfully loaded {len(self.stories)} stories into RAM.")

    def __len__(self):
        return len(self.stories)

    def __getitem__(self, idx):
        text = self.stories[idx]
        
        # Tokenize on the fly. PreTrainedTokenizerFast is heavily optimized in Rust,
        # so doing this per-item during training is extremely efficient.
        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.seq_len,
            padding="max_length",
            return_tensors="pt"
        )
        
        # Squeeze to remove the batch dimension added by the tokenizer
        return {"input_ids": encoding["input_ids"].squeeze(0)}

def get_tinystories_dataloader(file_path="tinystories_dataset.txt", batch_size=32, seq_len=256):
    """Loads local TinyStories text file and tokenizes with custom Word-Level tokenizer"""
    
    print("Initializing Custom Word-Level Tokenizer...")
    tokenizer = PreTrainedTokenizerFast(tokenizer_file="tinystories_wordlevel.json")
    
    # Explicitly map the special control tokens
    tokenizer.add_special_tokens({
        'unk_token': '[UNK]',
        'pad_token': '[PAD]',
        'bos_token': '[BOS]',
        'eos_token': '[EOS]'
    })
    
    vocab_size = len(tokenizer)
    
    # Initialize the custom dataset
    dataset = TinyStoriesLocalDataset(
        file_path=file_path, 
        tokenizer=tokenizer, 
        seq_len=seq_len, 
        limit=1000000
    )
    
    # Create the PyTorch DataLoader
    # Note: num_workers=2 or 4 can speed up data loading if your CPU allows it
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    
    return dataloader, vocab_size

In [4]:
import os
import torch
import torch.nn.functional as F
import wandb
from tqdm import tqdm

# ==========================================
# 4. Training Loop (Experiment B)
# ==========================================
def train():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    seq_len = 128
    
    batch_size = 32           
    accumulation_steps = 4    
    T_DIFFUSION = 2000
    EPOCHS = 1 
    
    N_RECURSIONS = 6
    T_CYCLES = 3
    
    dataloader, vocab_size = get_tinystories_dataloader(
        file_path="tinystories_dataset.txt", 
        batch_size=batch_size, 
        seq_len=seq_len
    )
    
    model = TRM_Diffusion(vocab_size=vocab_size, d_model=512, num_layers=4).to(device)
    model.token_emb.weight.requires_grad = False
    
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=0.01)
    ema = EMA(model, decay=0.999)
    
    wandb.init(
        project="trm-continuous-diffusion",
        name="Experiment-B-Deep-Recursion",
        config={
            "dataset_size": "1M",
            "epochs": EPOCHS,
            "effective_batch_size": batch_size * accumulation_steps,
            "micro_batch_size": batch_size,
            "seq_len": seq_len,
            "learning_rate": 1e-4,
            "N_recursions": N_RECURSIONS,
            "T_cycles": T_CYCLES
        }
    )
    
    print("Starting Experiment B: Deep Recursion with Syntax Anchor...")
    model.train()
    global_step = 0 
    optimizer.zero_grad()
    
    for epoch in range(EPOCHS): 
        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=True)
        
        for step, batch in enumerate(pbar):
            input_ids = batch["input_ids"].to(device)
            bs = input_ids.shape[0]
            
            y_true_emb = model.token_emb(input_ids)
            positions = torch.arange(seq_len, device=device).unsqueeze(0).expand(bs, seq_len)
            y_true_emb = y_true_emb + model.pos_emb(positions)
            
            t = torch.randint(1, T_DIFFUSION + 1, (bs,), device=device)
            alpha_bar_t = compute_sqrt_alpha_bar(t, T_max=T_DIFFUSION).view(bs, 1, 1)
            noise = torch.randn_like(y_true_emb)
            
            x_t_emb = torch.sqrt(alpha_bar_t) * y_true_emb + torch.sqrt(1 - alpha_bar_t) * noise
            
            # DEEP RECURSION FORWARD PASS
            # FIX 4: No tuple unpacking. forward() handles the z expansion internally now.
            pred_x0_emb = model(x_t_emb, t, positions, n=N_RECURSIONS, T=T_CYCLES)

            # Project to Discrete Vocabulary
            logits = model.lm_head(pred_x0_emb) 
            
            # Calculate Diffusion-LM Losses
            loss_recon = F.mse_loss(pred_x0_emb, y_true_emb)
            
            # FIX 5: Bulletproof flattening for Cross-Entropy
            loss_ce = F.cross_entropy(logits.view(-1, vocab_size), input_ids.reshape(-1))
            total_loss = loss_recon + (0.5 * loss_ce)
            
            # GRADIENT ACCUMULATION BACKPROPAGATION
            loss = total_loss / accumulation_steps
            loss.backward()
            
            if (step + 1) % accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                optimizer.zero_grad()
                ema.update() 
                
                wandb.log({
                    "train/total_loss": total_loss.item(),
                    "train/loss_recon": loss_recon.item(),
                    "train/loss_ce_discrete": loss_ce.item(),
                    "epoch": epoch,
                    "global_step": global_step
                })
                
                global_step += 1
                
                if global_step > 0 and global_step % 5000 == 0:
                    ema.apply_shadow()
                    save_path = f"trm_diffusion_step_{global_step}.pt"
                    torch.save({
                        'epoch': epoch,
                        'global_step': global_step,
                        'model_state_dict': model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                    }, save_path)
                    wandb.save(save_path) 
                    ema.restore()
            
            if step % 10 == 0:
                pbar.set_postfix({'Loss': f"{total_loss.item():.4f}"})

        print(f"--- Epoch {epoch+1} Complete ---")
        ema.apply_shadow()
        save_path = f"trm_diffusion_epoch_{epoch+1}_complete.pt"
        torch.save({
            'epoch': epoch,
            'global_step': global_step,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }, save_path)
        wandb.save(save_path) 
        ema.restore()
        
    wandb.finish()

if __name__ == "__main__":
    train()

Initializing Custom Word-Level Tokenizer...
Loading up to 1000000 stories from tinystories_dataset.txt...
Successfully loaded 999929 stories into RAM.


wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: rao-adnan-098 (rao-adnan-098-jamia-millia-islamia) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Starting Experiment B: Deep Recursion with Syntax Anchor...


Epoch 1/1: 100%|██████████| 31248/31248 [2:11:53<00:00,  3.95it/s, Loss=0.2594]  
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


--- Epoch 1 Complete ---


epoch,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
global_step,▁▁▁▁▁▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇█
train/loss_ce_discrete,█▆▅▅▅▅▃▃▃▂▃▂▃▂▃▂▃▃▂▂▃▃▂▂▁▂▂▁▃▂▂▂▂▁▂▂▂▃▂▂
train/loss_recon,█▆▆▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▂▁▂▁▂▁▁▂▂▂▁▂▁▁▁▁▁▁▁▁
train/total_loss,██▆▅▃▃▃▃▃▃▃▂▂▂▂▂▂▂▁▁▂▂▂▁▂▂▁▂▁▁▂▁▁▂▁▁▂▁▂▂
epoch,0
global_step,7811
train/loss_ce_discrete,0.22763
train/loss_recon,0.25169
train/total_loss,0.36551


In [4]:
import os
import torch
import torch.nn.functional as F
import wandb
from tqdm import tqdm

# ==========================================
# Phase 2: Semantic Fine-Tuning (Epoch 2-10)
# ==========================================
def train_phase_2():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    seq_len = 128
    
    # --- MEMORY SETTINGS ---
    batch_size = 64           
    accumulation_steps = 2    
    
    T_DIFFUSION = 2000
    TOTAL_EPOCHS = 10 
    START_EPOCH = 1 # 0-indexed, so 1 means we are starting at Epoch 2
    
    # --- DEEP RECURSION SETTINGS ---
    N_RECURSIONS = 6
    T_CYCLES = 3
    
    # 1. Load Custom TinyStories DataLoader
    dataloader, vocab_size = get_tinystories_dataloader(
        file_path="tinystories_dataset.txt", 
        batch_size=batch_size, 
        seq_len=seq_len
    )
    
    # 2. Initialize the Model
    model = TRM_Diffusion(vocab_size=vocab_size, d_model=512, num_layers=4).to(device)
    
    # 3. LOAD SAVED WEIGHTS FROM EPOCH 1
    checkpoint_path = "trm_diffusion_epoch_1_complete.pt"
    print(f"Loading checkpoint from {checkpoint_path}...")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    # Safely load state dict, handling both nested dictionary and direct state_dict formats
    state_dict = checkpoint.get('model_state_dict', checkpoint)
    model.load_state_dict(state_dict)
    print("Model weights successfully loaded!")
    
    # 4. THE CRITICAL FLIP: UNFREEZE THE EMBEDDINGS
    model.token_emb.weight.requires_grad = True
    print("Embeddings unfrozen! Model will now learn semantic grammar clusters.")
    
    # 5. Initialize a NEW Optimizer with a lower learning rate
    # We must do this because AdamW needs to track the newly unfrozen embeddings
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)
    
    ema = EMA(model, decay=0.999)
    
    wandb.init(
        project="trm-continuous-diffusion",
        name="Experiment-B-Phase-2-Unfrozen",
        config={
            "dataset_size": "1M",
            "epochs": TOTAL_EPOCHS,
            "starting_epoch": START_EPOCH + 1,
            "effective_batch_size": batch_size * accumulation_steps,
            "micro_batch_size": batch_size,
            "seq_len": seq_len,
            "learning_rate": 5e-5, # Log the dropped LR
            "N_recursions": N_RECURSIONS,
            "T_cycles": T_CYCLES,
            "embeddings_frozen": False
        }
    )
    
    print("Starting Phase 2 Training...")
    model.train()
    
    # Pick up the global step where Epoch 1 left off (roughly ~7800 steps for 1M stories at BS=128)
    global_step = checkpoint.get('global_step', 0)
    optimizer.zero_grad()
    
    for epoch in range(START_EPOCH, TOTAL_EPOCHS): 
        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{TOTAL_EPOCHS}", leave=True)
        
        for step, batch in enumerate(pbar):
            input_ids = batch["input_ids"].to(device)
            bs = input_ids.shape[0]
            
            # Clean x_0 Embeddings & Positional Anchor
            y_true_emb = model.token_emb(input_ids)
            positions = torch.arange(seq_len, device=device).unsqueeze(0).expand(bs, seq_len)
            y_true_emb = y_true_emb + model.pos_emb(positions)
            
            # Sample random timestep t and apply Sqrt Noise
            t = torch.randint(1, T_DIFFUSION + 1, (bs,), device=device)
            alpha_bar_t = compute_sqrt_alpha_bar(t, T_max=T_DIFFUSION).view(bs, 1, 1)
            noise = torch.randn_like(y_true_emb)
            
            # Forward Process q(x_t | x_0)
            x_t_emb = torch.sqrt(alpha_bar_t) * y_true_emb + torch.sqrt(1 - alpha_bar_t) * noise
            
            # DEEP RECURSION FORWARD PASS
            pred_x0_emb = model(x_t_emb, t, positions, n=N_RECURSIONS, T=T_CYCLES)

            # Project to Discrete Vocabulary
            logits = model.lm_head(pred_x0_emb) 
            
            # Calculate Diffusion-LM Losses
            loss_recon = F.mse_loss(pred_x0_emb, y_true_emb)
            loss_ce = F.cross_entropy(logits.view(-1, vocab_size), input_ids.view(-1))
            total_loss = loss_recon + (0.5 * loss_ce)
            
            # GRADIENT ACCUMULATION BACKPROPAGATION
            loss = total_loss / accumulation_steps
            loss.backward()
            
            if (step + 1) % accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                optimizer.zero_grad()
                ema.update() 
                
                wandb.log({
                    "train/total_loss": total_loss.item(),
                    "train/loss_recon": loss_recon.item(),
                    "train/loss_ce_discrete": loss_ce.item(),
                    "epoch": epoch,
                    "global_step": global_step
                })
                
                global_step += 1
                
                if global_step > 0 and global_step % 5000 == 0:
                    ema.apply_shadow()
                    save_path = f"trm_diffusion_phase2_step_{global_step}.pt"
                    torch.save({
                        'epoch': epoch,
                        'global_step': global_step,
                        'model_state_dict': model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                    }, save_path)
                    wandb.save(save_path) 
                    ema.restore()
            
            if step % 10 == 0:
                pbar.set_postfix({'Loss': f"{total_loss.item():.4f}"})

        print(f"--- Epoch {epoch+1} Complete ---")
        ema.apply_shadow()
        save_path = f"trm_diffusion_epoch_{epoch+1}_complete.pt"
        torch.save({
            'epoch': epoch,
            'global_step': global_step,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }, save_path)
        wandb.save(save_path) 
        ema.restore()
        
    wandb.finish()

if __name__ == "__main__":
    train_phase_2()

Initializing Custom Word-Level Tokenizer...
Loading up to 1000000 stories from tinystories_dataset.txt...
Successfully loaded 999929 stories into RAM.
Loading checkpoint from trm_diffusion_epoch_1_complete.pt...
Model weights successfully loaded!
Embeddings unfrozen! Model will now learn semantic grammar clusters.


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: rao-adnan-098 (rao-adnan-098-jamia-millia-islamia) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Starting Phase 2 Training...


Epoch 2/10: 100%|██████████| 15624/15624 [2:11:35<00:00,  1.98it/s, Loss=0.3550]
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


--- Epoch 2 Complete ---


...

Epoch 9/10: 100%|██████████| 15624/15624 [2:18:23<00:00,  1.88it/s, Loss=0.1505]
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


--- Epoch 9 Complete ---


Epoch 10/10: 100%|██████████| 15624/15624 [2:11:08<00:00,  1.99it/s, Loss=0.1032]
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


--- Epoch 10 Complete ---


epoch,▁▁▁▁▁▁▁▁▂▂▂▃▃▃▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇██
global_step,▁▁▁▂▂▂▃▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇████
train/loss_ce_discrete,█▄▇▇▃▇▇▇▄▅▅▅▃▅▅▄▄▆▆▂▇▃▇▂▃▄▆▄▃▄▃▃▄▂▄▁▂▇▂▁
train/loss_recon,█▆▆▅▅▅▄▆▄▃▄▃▃▃▃▂▂▂▃▂▂▃▂▂▂▂▂▁▂▁▂▂▂▁▁▂▁▁▂▂
train/total_loss,▆▆▆█▅▄▅▆▄▄▄▅▃▃▂▃▂▃▄▁▂▂▄▂▃▂▂▁▂▃▂▄▁▁▂▁▁▂▁▁
epoch,9
global_step,78119
train/loss_ce_discrete,0.15579
train/loss_recon,0.1176
train/total_loss,0.1955


Evaluation

In [3]:
import torch
import math
import json
from tqdm import tqdm
from transformers import PreTrainedTokenizerFast, GPT2LMHeadModel, GPT2Tokenizer
import nltk
from nltk.util import ngrams


nltk.download('punkt_tab', quiet=True)

# =============================================================================
# 1. GENERATION PIPELINE (Integrated from your training script)
# =============================================================================
def generate_samples(model_path, tokenizer_path, num_samples=100, seq_len=128, device="cuda"):
    print(f"--- 1. Generating {num_samples} samples from {model_path} ---")
    
    device = torch.device(device if torch.cuda.is_available() else "cpu")
    tokenizer = PreTrainedTokenizerFast(tokenizer_file=tokenizer_path)
    vocab_size = len(tokenizer)
    
    # IMPORTANT: Ensure TRM_Diffusion is imported or defined in this file
    # from your_model_file import TRM_Diffusion 
    
    # Initialize and load model
    model = TRM_Diffusion(vocab_size=vocab_size, d_model=512, num_layers=4).to(device)
    checkpoint = torch.load(model_path, map_location=device)
    state_dict = checkpoint.get('model_state_dict', checkpoint)
    model.load_state_dict(state_dict)
    model.eval()
    
    # Diffusion Parameters
    T_MAX = 2000
    inference_steps = 200 
    indices = torch.linspace(T_MAX, 1, inference_steps).long().to(device)
    
    results = []
    
    # Process in batches to avoid OOM on the L40S
    batch_size = 25 
    num_batches = math.ceil(num_samples / batch_size)

    # --- REMOVE THIS DUMMY DATA BLOCK WHEN RUNNING YOUR ACTUAL MODEL ---
    # print("[NOTE] Bypassing actual generation for script testing. Returning dummy data.")
    # return [
    #     "Once upon a time , there was a little girl named Lily . She loved going to the park .",
    #     "The dog ran fast . It was a big dog . The boy smiled and said hello .",
    #     "Once upon a time , a little boy named Ben saw a butterfly . He was happy ."
    # ] * (num_samples // 3)
    # -------------------------------------------------------------------

    
    # UNCOMMENT THIS BLOCK TO RUN ACTUAL INFERENCE
    with torch.no_grad():
        for b in tqdm(range(num_batches), desc="Generating Batches"):
            current_bs = min(batch_size, num_samples - b * batch_size)
            
            # Pure Gaussian noise initialization
            x_t = torch.randn((current_bs, seq_len, 512), device=device)
            positions = torch.arange(seq_len, device=device).unsqueeze(0).expand(current_bs, seq_len)

            for i in range(len(indices)):
                t = indices[i].expand(current_bs)
                
                # Predict x_0 from current x_t (n=6, T=3 for thesis architecture)
                pred_x0_emb = model(x_t, t, positions, n=6, T=3)
                
                # Deterministic Denoising step
                if i < len(indices) - 1:
                    alpha = 0.1 # Step size
                    x_t = (1 - alpha) * x_t + alpha * pred_x0_emb
                else:
                    x_t = pred_x0_emb

            # Decode final continuous vectors to discrete tokens
            logits = model.lm_head(x_t)
            sampled_ids = torch.argmax(logits, dim=-1)

            for i in range(current_bs):
                text = tokenizer.decode(sampled_ids[i], skip_special_tokens=True)
                results.append(text)
    
    return results

# =============================================================================
# 2. PSEUDO-PERPLEXITY (Fluency Evaluation)
# =============================================================================
def calculate_pseudo_perplexity(texts, device="cuda"):
    print("\n--- 2. Calculating Pseudo-Perplexity (Fluency) ---")
    print("Loading standard GPT-2 evaluator model...")
    
    eval_model_id = "gpt2" 
    eval_tokenizer = GPT2Tokenizer.from_pretrained(eval_model_id)
    eval_model = GPT2LMHeadModel.from_pretrained(eval_model_id).to(device)
    eval_model.eval()

    total_nll = 0.0
    total_length = 0

    with torch.no_grad():
        for text in tqdm(texts, desc="Evaluating PPL"):
            encodings = eval_tokenizer(text, return_tensors="pt")
            input_ids = encodings.input_ids.to(device)
            seq_len = input_ids.size(1)

            if seq_len < 2:
                continue

            outputs = eval_model(input_ids, labels=input_ids)
            neg_log_likelihood = outputs.loss * seq_len
            
            total_nll += neg_log_likelihood.item()
            total_length += seq_len

    avg_nll = total_nll / total_length
    ppl = math.exp(avg_nll)
    print(f">>> Final Pseudo-Perplexity: {ppl:.2f}")
    return ppl

# =============================================================================
# 3. N-GRAM DIVERSITY (Mode Collapse Evaluation)
# =============================================================================
def calculate_diversity_metrics(texts):
    print("\n--- 3. Calculating N-Gram Diversity (Mode Collapse) ---")
    
    all_bigrams = []
    all_trigrams = []
    
    for text in tqdm(texts, desc="Tokenizing N-Grams"):
        tokens = nltk.word_tokenize(text.lower())
        all_bigrams.extend(list(ngrams(tokens, 2)))
        all_trigrams.extend(list(ngrams(tokens, 3)))

    unique_bigrams = len(set(all_bigrams))
    total_bigrams = max(len(all_bigrams), 1)
    
    unique_trigrams = len(set(all_trigrams))
    total_trigrams = max(len(all_trigrams), 1)

    diversity_2 = unique_bigrams / total_bigrams
    diversity_3 = unique_trigrams / total_trigrams

    print(f">>> Unique Bi-grams:  {diversity_2:.2%} ({unique_bigrams}/{total_bigrams})")
    print(f">>> Unique Tri-grams: {diversity_3:.2%} ({unique_trigrams}/{total_trigrams})")
    
    return {"diversity_2gram": diversity_2, "diversity_3gram": diversity_3}

# =============================================================================
# MAIN EXECUTION
# =============================================================================
if __name__ == "__main__":
    device = "cuda" if torch.cuda.is_available() else "cpu"
    num_eval_samples = 500 # 500 is a solid sample size for academic benchmarks
    
    # 1. Generate text
    generated_texts = generate_samples(
        model_path="trm_diffusion_epoch_10_complete.pt",
        tokenizer_path="tinystories_wordlevel.json",
        num_samples=num_eval_samples, 
        device=device
    )
    
    # 2. Run Evaluations
    metrics = {
        "model_checkpoint": "trm_diffusion_epoch_10_complete.pt",
        "num_samples_evaluated": num_eval_samples
    }
    
    metrics["pseudo_perplexity"] = calculate_pseudo_perplexity(generated_texts, device)
    metrics.update(calculate_diversity_metrics(generated_texts))
    
    # 3. Save to JSON
    output_file = "eval_metrics_epoch_10.json"
    with open(output_file, "w") as f:
        json.dump(metrics, f, indent=4)
        
    print("\n==================================================")
    print(f"Evaluation Complete! Results saved to {output_file}")
    print("==================================================")

--- 1. Generating 500 samples from trm_diffusion_epoch_10_complete.pt ---


Generating Batches: 100%|██████████| 20/20 [04:49<00:00, 14.48s/it]



--- 2. Calculating Pseudo-Perplexity (Fluency) ---
Loading standard GPT-2 evaluator model...


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Evaluating PPL: 100%|██████████| 500/500 [00:02<00:00, 203.84it/s]


>>> Final Pseudo-Perplexity: 85.29

--- 3. Calculating N-Gram Diversity (Mode Collapse) ---


Tokenizing N-Grams: 100%|██████████| 500/500 [00:00<00:00, 2296.72it/s]

>>> Unique Bi-grams:  17.97% (11308/62932)
>>> Unique Tri-grams: 54.46% (34000/62432)

Evaluation Complete! Results saved to eval_metrics_epoch_10.json


In [5]:
import torch
import time
import json
import math
from tqdm import tqdm
from transformers import PreTrainedTokenizerFast, GPT2LMHeadModel, GPT2Tokenizer

# =============================================================================
# HELPER: PSEUDO-PERPLEXITY EVALUATOR
# =============================================================================
def calculate_pseudo_perplexity(texts, device="cuda"):
    """Quick PPL calculator using standard GPT-2"""
    eval_model_id = "gpt2" 
    eval_tokenizer = GPT2Tokenizer.from_pretrained(eval_model_id)
    eval_model = GPT2LMHeadModel.from_pretrained(eval_model_id).to(device)
    eval_model.eval()

    total_nll = 0.0
    total_length = 0

    with torch.no_grad():
        for text in texts:
            encodings = eval_tokenizer(text, return_tensors="pt")
            input_ids = encodings.input_ids.to(device)
            seq_len = input_ids.size(1)

            if seq_len < 2: continue

            outputs = eval_model(input_ids, labels=input_ids)
            total_nll += (outputs.loss * seq_len).item()
            total_length += seq_len

    # Clean up VRAM after evaluation
    del eval_model
    torch.cuda.empty_cache()
    
    return math.exp(total_nll / total_length)

# =============================================================================
# EXPERIMENT 1: THE DENOISING TRAJECTORY ("Waterfall Plot")
# =============================================================================
def experiment_trajectory(model, tokenizer, device, seq_len=128):
    print("\n" + "="*50)
    print("EXPERIMENT 1: DENOISING TRAJECTORY (LATENT WATERFALL)")
    print("="*50)
    
    T_MAX = 2000
    inference_steps = 200
    indices = torch.linspace(T_MAX, 1, inference_steps).long().to(device)
    
    # We want to take snapshots at these specific steps remaining
    snapshots = [200, 150, 100, 50, 1] 
    
    with torch.no_grad():
        x_t = torch.randn((1, seq_len, 512), device=device)
        positions = torch.arange(seq_len, device=device).unsqueeze(0)

        for i, t_val in enumerate(indices):
            t = t_val.unsqueeze(0)
            steps_remaining = inference_steps - i
            
            # Predict x0
            pred_x0_emb = model(x_t, t, positions, n=6, T=3)
            
            # Take a snapshot if we hit our target steps
            if steps_remaining in snapshots:
                logits = model.lm_head(pred_x0_emb)
                sampled_ids = torch.argmax(logits, dim=-1)
                text = tokenizer.decode(sampled_ids[0], skip_special_tokens=True)
                print(f"\n[Step {steps_remaining:03d} | High Noise]" if steps_remaining > 100 else f"\n[Step {steps_remaining:03d} | Low Noise]")
                print(f"-> {text}")

            # Denoise step
            if i < len(indices) - 1:
                alpha = 0.1 
                x_t = (1 - alpha) * x_t + alpha * pred_x0_emb
            else:
                x_t = pred_x0_emb

# =============================================================================
# EXPERIMENT 2: INFERENCE LATENCY BENCHMARKS
# =============================================================================
def experiment_latency(model, device, batch_size=50, seq_len=128):
    print("\n" + "="*50)
    print(f"EXPERIMENT 2: LATENCY BENCHMARK (Batch Size: {batch_size})")
    print("="*50)
    
    step_counts = [20, 50, 100, 200]
    results = {}
    
    # Warmup GPU (Standard practice for accurate PyTorch timing)
    print("Warming up GPU...")
    dummy_x = torch.randn((batch_size, seq_len, 512), device=device)
    dummy_t = torch.tensor([1000]*batch_size, device=device)
    dummy_pos = torch.arange(seq_len, device=device).unsqueeze(0).expand(batch_size, seq_len)
    with torch.no_grad():
        for _ in range(5):
            _ = model(dummy_x, dummy_t, dummy_pos, n=6, T=3)
    torch.cuda.synchronize()

    with torch.no_grad():
        for steps in step_counts:
            indices = torch.linspace(2000, 1, steps).long().to(device)
            x_t = torch.randn((batch_size, seq_len, 512), device=device)
            
            # Start timer
            torch.cuda.synchronize()
            start_time = time.time()
            
            for i, t_val in enumerate(indices):
                t = t_val.expand(batch_size)
                pred_x0_emb = model(x_t, t, dummy_pos, n=6, T=3)
                if i < len(indices) - 1:
                    x_t = (1 - 0.1) * x_t + 0.1 * pred_x0_emb
                    
            torch.cuda.synchronize()
            end_time = time.time()
            
            total_time = end_time - start_time
            time_per_sample = total_time / batch_size
            
            print(f"Steps: {steps:<4} | Total Time: {total_time:.2f}s | Latency per sample: {time_per_sample:.3f}s")
            results[steps] = time_per_sample
            
    return results

# =============================================================================
# EXPERIMENT 3: ABLATION STUDY (STEPS VS. PPL)
# =============================================================================
def experiment_ablation(model, tokenizer, device, num_samples=100, seq_len=128):
    print("\n" + "="*50)
    print("EXPERIMENT 3: ABLATION STUDY (PPL vs STEPS)")
    print("="*50)
    
    step_counts = [20, 50, 100, 200]
    ablation_results = {}
    batch_size = 50
    num_batches = math.ceil(num_samples / batch_size)
    
    for steps in step_counts:
        print(f"\nGenerating {num_samples} samples with {steps} diffusion steps...")
        indices = torch.linspace(2000, 1, steps).long().to(device)
        generated_texts = []
        
        with torch.no_grad():
            for _ in range(num_batches):
                x_t = torch.randn((batch_size, seq_len, 512), device=device)
                positions = torch.arange(seq_len, device=device).unsqueeze(0).expand(batch_size, seq_len)
                
                for i, t_val in enumerate(indices):
                    t = t_val.expand(batch_size)
                    pred_x0_emb = model(x_t, t, positions, n=6, T=3)
                    if i < len(indices) - 1:
                        x_t = (1 - 0.1) * x_t + 0.1 * pred_x0_emb
                    else:
                        x_t = pred_x0_emb
                        
                logits = model.lm_head(x_t)
                sampled_ids = torch.argmax(logits, dim=-1)
                for j in range(batch_size):
                    generated_texts.append(tokenizer.decode(sampled_ids[j], skip_special_tokens=True))
                    
        # Calculate PPL for this step count
        print(f"Evaluating PPL for {steps} steps...")
        ppl = calculate_pseudo_perplexity(generated_texts, device)
        print(f"-> {steps} Steps PPL: {ppl:.2f}")
        ablation_results[steps] = ppl
        
    return ablation_results

# =============================================================================
# MAIN EXECUTION
# =============================================================================
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Starting Thesis Experiments on: {device.type.upper()}")
    
    # 1. Setup Tokenizer & Model
    tokenizer_path = "/workspace/Diffusion_TRM/tinystories_wordlevel.json"
    model_path = "/workspace/Diffusion_TRM/trm_diffusion_epoch_10_complete.pt"
    tokenizer = PreTrainedTokenizerFast(tokenizer_file=tokenizer_path)
    
    # IMPORTANT: Initialize your TRM_Diffusion model here
    # from your_model_file import TRM_Diffusion
    model = TRM_Diffusion(vocab_size=len(tokenizer), d_model=512, num_layers=4).to(device)
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    # --- DUMMY BYPASS FOR SCRIPT TESTING (REMOVE WHEN RUNNING) ---
    print("\n[WARNING: Model initialization bypassed. Add your TRM load logic above to run actual tests.]")
    # -------------------------------------------------------------
    
    # Uncomment these to run the actual experiments once your model is loaded!

    # Run Experiment 1
    experiment_trajectory(model, tokenizer, device)
    
    # Run Experiment 2
    latency_data = experiment_latency(model, device)
    
    # Run Experiment 3
    ablation_data = experiment_ablation(model, tokenizer, device)
    
    # Save Final Data
    final_report = {
        "latency_seconds_per_sample": latency_data,
        "ablation_ppl_by_steps": ablation_data
    }
    with open("thesis_experiments_report.json", "w") as f:
        json.dump(final_report, f, indent=4)
    print("\nAll experiments complete! Data saved to thesis_experiments_report.json")
    

Starting Thesis Experiments on: CUDA



[WARNING: Model initialization bypassed. Add your TRM load logic above to run actual tests.]

EXPERIMENT 1: DENOISING TRAJECTORY (LATENT WATERFALL)

[Step 200 | High Noise]
-> Once upon a time , there was a little girl named . . She loved to play with . . and . . . , . to . . . . . , , . , . . . the . . . . . the . . . . . . . . . . . . . . . . , and the , . . . . and . . a . . the . . the . . . . . , . . . . . . . . . . . was . . . . . . . . . . . . . . . . . . . , , . . . . . .

[Step 150 | High Noise]
-> Once upon a time , there was a little boy named Timmy . He loved to play with a truck and wear . He always liked to his head . One day , he was a big truck in the fence . He loved a big truck to talk to jump that he saw Timmy in the sky and his head . He asked the truck was wrong . Timmy and he saw a truck that he saw a big magical truck . The truck ! , " No ! Can I ' s here . was He back to and hugged his head , Timmy used his s backyard . Timmy bright and pretty . They could able

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


-> 20 Steps PPL: 502.54

Generating 100 samples with 50 diffusion steps...
Evaluating PPL for 50 steps...


...

-> 50 Steps PPL: 77.11

Generating 100 samples with 100 diffusion steps...
Evaluating PPL for 100 steps...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

-> 100 Steps PPL: 146.61

Generating 100 samples with 200 diffusion steps...
Evaluating PPL for 200 steps...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

-> 200 Steps PPL: 85.86

All experiments complete! Data saved to thesis_experiments_report.json


Prompt

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import GPT2Tokenizer
import math
import copy

# ==========================================
# 1. Diffusion Utilities (Sqrt Schedule)
# ==========================================
def compute_sqrt_alpha_bar(t, T_max=2000, s=1e-4):
    """
    Sqrt noise schedule from Diffusion-LM (Appendix A).
    alpha_bar_t = 1 - sqrt(t/T + s)
    """
    # t is expected to be a tensor of shape (batch_size, 1, 1)
    t_norm = t.float() / T_max
    alpha_bar = 1.0 - torch.sqrt(t_norm + s)
    # Clamp to prevent negative variances or exact zeros
    return torch.clamp(alpha_bar, min=1e-5, max=0.9999)

class EMA:
    """Exponential Moving Average for model weights (Crucial for TRM stability)"""
    def __init__(self, model, decay=0.999):
        self.model = model
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    def update(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                new_average = (1.0 - self.decay) * param.data + self.decay * self.shadow[name]
                self.shadow[name] = new_average.clone()

    def apply_shadow(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data
                param.data = self.shadow[name]

    def restore(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                param.data = self.backup[name]

In [7]:
import torch
import torch.nn.functional as F
from transformers import GPT2Tokenizer
import math

# 1. Include your TRM_Diffusion class and BidirectionalTransformerLayer here
# (Copy them from your training script so the model can be instantiated)
# class BidirectionalTransformerLayer(nn.Module): ...
# class TRM_Diffusion(nn.Module): ...

def compute_sqrt_alpha_bar(t, T_max=2000, s=1e-4):
    """Sqrt noise schedule (matches the training script)"""
    t_norm = t.float() / T_max
    alpha_bar = 1.0 - torch.sqrt(t_norm + s)
    return torch.clamp(alpha_bar, min=1e-5, max=0.9999)

def clamp_to_nearest_word(pred_x0_emb, vocab_embeddings):
    """
    THE CLAMPING TRICK:
    Finds the closest actual word embedding for each predicted continuous vector.
    Using L2 distance: ||pred - vocab||^2
    """
    # pred_x0_emb shape: (batch_size, seq_len, d_model)
    # vocab_embeddings shape: (vocab_size, d_model)
    
    # Calculate Euclidean distance between predictions and all vocab words
    dists = torch.cdist(pred_x0_emb, vocab_embeddings, p=2) # shape: (batch, seq, vocab)
    
    # Find the index of the closest word for each position
    nearest_idx = dists.argmin(dim=-1) 
    
    # Snap the vector to exactly that word's embedding
    clamped_x0_emb = vocab_embeddings[nearest_idx]
    
    return clamped_x0_emb, nearest_idx

In [14]:
import torch
from transformers import PreTrainedTokenizerFast

@torch.no_grad()
def generate_with_prompt(model_path, prompt_text, seq_len=128, inference_steps=100, device="cuda"):
    print("\n" + "="*50)
    print(f"GENERATING WITH PROMPT: '{prompt_text}'")
    print("="*50)

    device = torch.device(device if torch.cuda.is_available() else "cpu")
    
    # 1. Setup Tokenizer & Model
    tokenizer = PreTrainedTokenizerFast(tokenizer_file="tinystories_wordlevel.json")
    vocab_size = len(tokenizer)
    
    # IMPORTANT: Ensure TRM_Diffusion is imported or defined in this file
    # from your_model_file import TRM_Diffusion
    
    # Initialize Model
    model = TRM_Diffusion(vocab_size=vocab_size, d_model=512, num_layers=4).to(device)
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    # 2. Extract Exact Embeddings for the Prompt
    prompt_tokens = tokenizer(prompt_text, return_tensors="pt")["input_ids"].to(device)
    prompt_len = prompt_tokens.shape[1]
    
    if prompt_len >= seq_len:
        raise ValueError(f"Prompt length ({prompt_len}) exceeds total sequence length ({seq_len}).")
        
    positions = torch.arange(seq_len, device=device).unsqueeze(0)
    
    # Calculate the mathematically perfect continuous vector for the prompt
    clean_prompt_emb = model.token_emb(prompt_tokens) + model.pos_emb(positions[:, :prompt_len])
    
    print(f"Prompt length: {prompt_len} tokens. Generating remaining {seq_len - prompt_len} tokens...\n")
    
    # 3. Diffusion Loop (Optimized Deterministic Sampling)
    # Start with pure Gaussian noise
    x_t = torch.randn((1, seq_len, 512), device=device)
    indices = torch.linspace(2000, 1, inference_steps).long().to(device)
    alpha = 0.1 # Linear interpolation step size
    
    for i, t_val in enumerate(indices):
        t = t_val.unsqueeze(0)
        
        # Let the TRM predict the clean sequence
        pred_x0_emb = model(x_t, t, positions, n=6, T=3)
        
        # --- LATENT INPAINTING ---
        # Force the beginning of the prediction to exactly match our prompt
        pred_x0_emb[:, :prompt_len, :] = clean_prompt_emb
        
        # Step x_t towards the anchored prediction
        if i < len(indices) - 1:
            x_t = (1 - alpha) * x_t + alpha * pred_x0_emb
        else:
            x_t = pred_x0_emb

    # 4. Decode the final discrete tokens
    logits = model.lm_head(x_t)
    sampled_ids = torch.argmax(logits, dim=-1)
    
    final_text = tokenizer.decode(sampled_ids[0], skip_special_tokens=True)
    
    print("--- Final Generated Story ---")
    print(final_text)
    print("==================================================")
    
    return final_text

# ==========================================
# Example Usage:
# ==========================================
if __name__ == "__main__":
    generate_with_prompt(
        model_path="trm_diffusion_epoch_10_complete.pt", 
        prompt_text="Once upon a time , a little dog named Buster",
        seq_len=128,
        inference_steps=200 # Using the optimized step count from your ablation study
    )


GENERATING WITH PROMPT: 'Once upon a time , a little dog named Buster'
Prompt length: 10 tokens. Generating remaining 118 tokens...



--- Final Generated Story ---
Once upon a time , a little dog named day . It had a to pet , Tim saw a big friend . She wanted to show her squirrel wanted to play with the squirrel . " I wanted to play outside and said . saw the squirrel . Her young squirrel said , " the dog and the said , " puppy !" ' s friend . She was wrong and said , " Mommy , " I can draw ." she went inside . You both Tim and said , " Thank you play at the toy squirrel , a squirrel . She said " Hello , I will ask the new tree with you if you . Tim was playing with her
